## Cluster Embeddings for W3 Ground Truth (text-embedding-3-small)

Clusters text-embedding-3-small embeddings within each category, separately for abstracts and PDF chunks, to build the semantically-grounded ground truth needed for W3 (semantic similarity workload) queries.

**Why cluster within category rather than globally**: your abstract categories (cs.AI, cs.IR, etc.) are broad author-assigned labels. A single category can contain several genuinely distinct sub-themes (e.g. cs.IR contains dense retrieval, recommendation systems, query understanding as different sub-topics that happen to share one arxiv tag). Clustering within each category surfaces these tighter sub-themes so a W3 query like "papers about efficient attention mechanisms" has a real, coherent relevant-set behind it, rather than an entire noisy category. Clustering globally across categories was considered and rejected: it would mix categories together and lose the category-wise structure your whole benchmark is built around (query generation, database ingestion, and results are all reported per category).

**Two corpora, run separately, never merged**: abstracts are one embedding per paper (clean unit for clustering). PDF chunks are one embedding per page, so a single long paper can contribute many page-level points. These have different statistical properties (page-chunks from the same paper are highly self-similar, which can bias cluster shapes toward long papers). Running them as two separate clustering passes, rather than one combined pass, keeps that difference visible and avoids silently mixing granularities into one set of cluster labels.

**Pipeline shape**:
1. Load embeddings for a category (from Hugging Face, uploaded in the prior embedding-generation step)
2. Normalize vectors (so Euclidean k-means behaves like cosine k-means)
3. Sweep k = 3 to 10, score each with silhouette score
4. Pick the best k per category, keep the full sweep results for transparency (not just the winner)
5. Optionally run HDBSCAN on the same category as a comparison, since no k needs to be chosen for it
6. Export cluster assignments as a new field per chunk_id, uploaded back to the Hub
7. Export a human-readable sample-per-cluster file for manual spot-checking (not scored automatically, meant for you to read)

### Install dependencies (Colab only, skip if already installed locally)

`hdbscan` is separate from `scikit-learn` and needs its own install. If it fails to import later, the notebook still runs the k-means portion, HDBSCAN is explicitly a comparison, not a requirement.

In [6]:
# Uncomment if running in Colab or a fresh environment
!pip install scikit-learn hdbscan huggingface_hub numpy -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Configuration

In [2]:
import os

# --- Hugging Face repo (shared corpus) ---
REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"
REPO_TYPE = "dataset"

# --- Which model's embeddings to cluster ---
EMBEDDING_MODEL = "text-embedding-3-small"

# --- The 10 target categories ---
ALL_CATEGORIES = [
    "cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE",       # yours
    "cs.CV", "cs.CL", "cs.NE", "cs.DC", "cs.CR",       # collaborator's
]

# --- k-means sweep range ---
K_MIN = 3
K_MAX = 10   # inclusive
RANDOM_STATE = 42  # fixed seed, so re-running reproduces the same cluster assignments

# --- HDBSCAN comparison ---
RUN_HDBSCAN_COMPARISON = True
HDBSCAN_MIN_CLUSTER_SIZE = 5  # smallest group HDBSCAN will call a cluster rather than noise

# --- Spot-check export ---
N_SAMPLES_PER_CLUSTER = 5  # how many example chunks to export per cluster for manual review

# --- Local paths ---
PROJECT_ROOT = ".."
CLUSTERS_DIR = os.path.join(PROJECT_ROOT, "data", "clusters", EMBEDDING_MODEL)
SPOTCHECK_DIR = os.path.join(PROJECT_ROOT, "data", "clusters", EMBEDDING_MODEL, "spot_check")
for corpus_type in ["abstracts", "pdf_chunks"]:
    os.makedirs(os.path.join(CLUSTERS_DIR, corpus_type), exist_ok=True)
os.makedirs(SPOTCHECK_DIR, exist_ok=True)

FORCE_RECOMPUTE = False  # set True to redo clustering for categories already clustered

### Authenticate with Hugging Face

In [3]:
from huggingface_hub import whoami, login

try:
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")
except Exception:
    print("Not authenticated yet, prompting for token...")
    login()
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")

c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authenticated as: Sakhiur


### Loader functions

Pulls embeddings (produced in the prior notebook) and, separately, the original source records (needed later for the spot-check export, since embeddings alone don't carry human-readable text). Both are joined on `chunk_id`.

In [4]:
import json
import numpy as np
from huggingface_hub import hf_hub_download


def load_embeddings(category: str, corpus_type: str) -> dict[str, list[float]]:
    """Returns {chunk_id: embedding_vector}. Empty dict if not found on the Hub."""
    repo_path = f"embeddings/{EMBEDDING_MODEL}/{corpus_type}/{category}.jsonl"
    try:
        local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
    except Exception as e:
        print(f"  [SKIP] {repo_path} not found on Hub ({e})")
        return {}

    result = {}
    with open(local_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            result[record["chunk_id"]] = record["embedding"]
    return result


def load_source_records(category: str, corpus_type: str) -> dict[str, dict]:
    """
    Returns {chunk_id: source_record} for spot-check display purposes.
    chunk_id construction mirrors the embedding-generation notebook exactly,
    so the two can be joined without ambiguity.
    """
    if corpus_type == "abstracts":
        repo_path = f"abstracts/by_category/{category}.jsonl"
    else:
        repo_path = f"pdf_chunks/{category}.jsonl"

    try:
        local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
    except Exception:
        return {}

    result = {}
    with open(local_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if corpus_type == "abstracts":
                chunk_id = record["id"]
            else:
                chunk_id = f"{record['paper_id']}_p{record['page_num']}"
            result[chunk_id] = record
    return result

### Clustering functions

**Why normalize before k-means**: OpenAI embeddings are approximately unit-length already, but normalizing explicitly guarantees it. On unit-normalized vectors, Euclidean distance and cosine distance produce the same relative ordering, so standard k-means (which minimizes Euclidean distance) behaves equivalently to a cosine-based clustering without needing a custom implementation. Skipping this step risks k-means responding to vector magnitude differences that don't reflect semantic difference.

**Why sweep k rather than pick one value**: no principled way to know the right number of sub-themes in a category ahead of time, it varies by category (a broad category like cs.LG likely has more distinct sub-themes than a narrower one like cs.DB). Silhouette score gives an empirical signal for which k best separates the data, per category, rather than assuming one k fits all ten categories.

**Why silhouette score specifically**: it's bounded in [-1, 1], doesn't require ground-truth labels (you don't have any, that's the whole point of this exercise), and penalizes both weak inter-cluster separation and weak intra-cluster cohesion in one number, making it a reasonable single metric to compare k values against each other within a category. It is not, on its own, sufficient proof of cluster meaningfulness, which is why the manual spot-check step exists downstream, silhouette score can be reasonably high even for clusters that don't mean anything to a human reader, so it's a necessary but not sufficient check.

In [8]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score


def sweep_kmeans(vectors: np.ndarray, k_min: int, k_max: int, random_state: int) -> dict:
    """
    Runs k-means for k in [k_min, k_max], returns per-k silhouette scores and labels.
    k values >= n_samples are skipped (k-means cannot produce more clusters than points).
    """
    n_samples = vectors.shape[0]
    results = {}

    for k in range(k_min, k_max + 1):
        if k >= n_samples:
            print(f"    [SKIP] k={k}: not enough samples ({n_samples})")
            continue

        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = km.fit_predict(vectors)

        # silhouette_score requires at least 2 distinct labels present
        if len(set(labels)) < 2:
            print(f"    [SKIP] k={k}: degenerate result, all points in one cluster")
            continue

        score = silhouette_score(vectors, labels, metric="euclidean")
        results[k] = {"labels": labels, "silhouette": score}
        print(f"    k={k}: silhouette={score:.4f}")

    return results


def pick_best_k(sweep_results: dict) -> int:
    """Selects the k with the highest silhouette score from a completed sweep."""
    return max(sweep_results.keys(), key=lambda k: sweep_results[k]["silhouette"])

### HDBSCAN comparison function

**Why compare against HDBSCAN specifically, not another method**: HDBSCAN doesn't require choosing k upfront, it infers the number of clusters from density structure in the data, and explicitly labels low-density points as noise (label -1) rather than forcing every point into a cluster the way k-means does. This is a meaningfully different assumption about what a "cluster" is, so comparing the two isn't just running two libraries for variety, it's testing whether your category's sub-themes are naturally density-separated (favoring HDBSCAN) or more evenly spread (favoring k-means's partition-based approach). Reporting both, with reasoning for which fits better per category, is what turns this from "we ran a clustering algorithm" into an actual small methods comparison worth a paragraph in the paper.

**Silhouette score caveat for HDBSCAN**: noise points (label -1) are excluded before scoring, since silhouette score is undefined for a "non-cluster" label and including them would distort the metric. This means an HDBSCAN silhouette score isn't perfectly apples-to-apples with a k-means score computed over 100% of the points, worth stating explicitly in the paper rather than presenting the two numbers as directly comparable without caveat.

In [10]:
def run_hdbscan(vectors: np.ndarray, min_cluster_size: int) -> dict | None:
    """
    Returns {'labels': ..., 'silhouette': ..., 'n_clusters': ..., 'n_noise': ...} or None if
    hdbscan isn't installed or the result is degenerate (all noise / single cluster).
    """
    try:
        import hdbscan
    except ImportError:
        print("    [SKIP] hdbscan not installed, skipping comparison for this category")
        return None

    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean")
    labels = clusterer.fit_predict(vectors)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int(np.sum(labels == -1))

    if n_clusters < 2:
        print(f"    [SKIP] HDBSCAN found {n_clusters} cluster(s), too degenerate to score")
        return {"labels": labels, "silhouette": None, "n_clusters": n_clusters, "n_noise": n_noise}

    non_noise_mask = labels != -1
    score = silhouette_score(vectors[non_noise_mask], labels[non_noise_mask], metric="euclidean")

    print(f"    HDBSCAN: {n_clusters} clusters, {n_noise}/{len(labels)} noise points, silhouette={score:.4f} (excl. noise)")
    return {"labels": labels, "silhouette": score, "n_clusters": n_clusters, "n_noise": n_noise}

### Run clustering per category, per corpus type

Loops both `abstracts` and `pdf_chunks` for all 10 categories. Categories missing from the Hub (your collaborator's, until he uploads) are skipped and reported, not treated as failures. Results (best-k labels, full sweep scores, HDBSCAN comparison) are kept in memory here and written to disk in the next cell, so a single dict holds everything needed for both the upload step and the verification step later.

In [11]:
all_results = {}  # {(corpus_type, category): {chunk_ids, kmeans_sweep, best_k, hdbscan_result}}

for corpus_type in ["abstracts", "pdf_chunks"]:
    for category in ALL_CATEGORIES:
        print(f"\n=== {corpus_type} / {category} ===")

        embeddings_dict = load_embeddings(category, corpus_type)
        if not embeddings_dict:
            continue

        chunk_ids = list(embeddings_dict.keys())
        vectors = np.array([embeddings_dict[cid] for cid in chunk_ids], dtype=np.float32)
        vectors = normalize(vectors, norm="l2")

        print(f"  {len(chunk_ids)} vectors, dim={vectors.shape[1]}")

        print("  k-means sweep:")
        sweep = sweep_kmeans(vectors, K_MIN, K_MAX, RANDOM_STATE)
        if not sweep:
            print(f"  [SKIP] {corpus_type}/{category}: no valid k in sweep range for this sample size")
            continue

        best_k = pick_best_k(sweep)
        print(f"  best k = {best_k} (silhouette={sweep[best_k]['silhouette']:.4f})")

        hdbscan_result = None
        if RUN_HDBSCAN_COMPARISON:
            print("  HDBSCAN comparison:")
            hdbscan_result = run_hdbscan(vectors, HDBSCAN_MIN_CLUSTER_SIZE)

        all_results[(corpus_type, category)] = {
            "chunk_ids": chunk_ids,
            "kmeans_sweep": sweep,
            "best_k": best_k,
            "hdbscan_result": hdbscan_result,
        }


=== abstracts / cs.AI ===


c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sakhiur\.cache\huggingface\hub\datasets--Sakhiur--empirical-rag-paradigm-benchmark. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  7369 vectors, dim=1536
  k-means sweep:
    k=3: silhouette=0.0260
    k=4: silhouette=0.0237
    k=5: silhouette=0.0274
    k=6: silhouette=0.0298
    k=7: silhouette=0.0295
    k=8: silhouette=0.0276
    k=9: silhouette=0.0288
    k=10: silhouette=0.0277
  best k = 6 (silhouette=0.0298)
  HDBSCAN comparison:
    HDBSCAN: 2 clusters, 3327/7369 noise points, silhouette=0.0487 (excl. noise)

=== abstracts / cs.LG ===
  10746 vectors, dim=1536
  k-means sweep:
    k=3: silhouette=0.0222
    k=4: silhouette=0.0219
    k=5: silhouette=0.0249
    k=6: silhouette=0.0242
    k=7: silhouette=0.0260
    k=8: silhouette=0.0283
    k=9: silhouette=0.0298
    k=10: silhouette=0.0309
  best k = 10 (silhouette=0.0309)
  HDBSCAN comparison:
    HDBSCAN: 11 clusters, 6571/10746 noise points, silhouette=-0.0374 (excl. noise)

=== abstracts / cs.IR ===
  3076 vectors, dim=1536
  k-means sweep:
    k=3: silhouette=0.0274
    k=4: silhouette=0.0281
    k=5: silhouette=0.0254
    k=6: silhouette=0.0260
 

KeyboardInterrupt: 

### Save cluster assignments locally

One JSONL per category per corpus type, one line per chunk_id, with three fields: `chunk_id`, `kmeans_cluster` (from the best-k run), `hdbscan_cluster` (or `null` if HDBSCAN wasn't run or the category was skipped). Keeping both algorithms' assignments side by side, rather than only the "winner," preserves the comparison for the paper and lets you decide later, per category, which one you trust more for that category's W3 ground truth.

In [ ]:
for (corpus_type, category), result in all_results.items():
    out_path = os.path.join(CLUSTERS_DIR, corpus_type, f"{category}.jsonl")

    if os.path.exists(out_path) and not FORCE_RECOMPUTE:
        print(f"[SKIP] {corpus_type}/{category}: already saved at {out_path}")
        continue

    chunk_ids = result["chunk_ids"]
    best_k = result["best_k"]
    kmeans_labels = result["kmeans_sweep"][best_k]["labels"]

    hdbscan_labels = None
    if result["hdbscan_result"] is not None:
        hdbscan_labels = result["hdbscan_result"]["labels"]

    with open(out_path, "w", encoding="utf-8") as f:
        for i, cid in enumerate(chunk_ids):
            record = {
                "chunk_id": cid,
                "kmeans_cluster": int(kmeans_labels[i]),
                "kmeans_k": int(best_k),
                "hdbscan_cluster": int(hdbscan_labels[i]) if hdbscan_labels is not None else None,
            }
            f.write(json.dumps(record) + "\n")

    print(f"Saved {out_path}")

### Export spot-check samples for manual review

**Why export to a file instead of scoring automatically**: silhouette score tells you the clusters are geometrically separated, it says nothing about whether a human would recognize them as a coherent topic. There's no automated substitute for actually reading a handful of titles/texts per cluster and judging "does this look like one theme." This step produces a readable file (grouped by cluster, a few examples each) specifically so you can do that judgment call yourself, per category, before trusting any cluster as W3 ground truth.

For abstracts, the `title` field is shown (short, fast to scan). For PDF chunks, the first ~200 characters of `text` are shown instead, since PDF chunks don't have a title field of their own.

In [ ]:
import random

random.seed(RANDOM_STATE)

for (corpus_type, category), result in all_results.items():
    source_records = load_source_records(category, corpus_type)
    if not source_records:
        print(f"[SKIP] {corpus_type}/{category}: could not reload source records for spot-check")
        continue

    chunk_ids = result["chunk_ids"]
    best_k = result["best_k"]
    kmeans_labels = result["kmeans_sweep"][best_k]["labels"]

    clusters = {}
    for cid, label in zip(chunk_ids, kmeans_labels):
        clusters.setdefault(int(label), []).append(cid)

    out_path = os.path.join(SPOTCHECK_DIR, f"{corpus_type}_{category}_kmeans_k{best_k}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"Spot-check samples: {corpus_type} / {category}, k-means k={best_k}\n")
        f.write(f"Silhouette score: {result['kmeans_sweep'][best_k]['silhouette']:.4f}\n")
        f.write("=" * 70 + "\n\n")

        for cluster_id in sorted(clusters.keys()):
            members = clusters[cluster_id]
            sample = random.sample(members, min(N_SAMPLES_PER_CLUSTER, len(members)))

            f.write(f"--- Cluster {cluster_id} ({len(members)} members total, showing {len(sample)}) ---\n")
            for cid in sample:
                record = source_records.get(cid)
                if record is None:
                    f.write(f"  [{cid}] (source record not found)\n")
                    continue
                if corpus_type == "abstracts":
                    f.write(f"  [{cid}] {record.get('title', '(no title)')}\n")
                else:
                    preview = record.get("text", "")[:200].replace("\n", " ")
                    f.write(f"  [{cid}] {preview}...\n")
            f.write("\n")

    print(f"Wrote spot-check file: {out_path}")

### Upload cluster assignments back to Hugging Face

Stored under `clusters/<model_name>/<corpus_type>/<category>.jsonl`, mirroring the embeddings path structure. Spot-check files are kept local only (they're a manual review aid, not pipeline input, no reason to version them on the Hub).

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

for corpus_type in ["abstracts", "pdf_chunks"]:
    local_dir = os.path.join(CLUSTERS_DIR, corpus_type)
    for fname in sorted(os.listdir(local_dir)):
        if not fname.endswith(".jsonl"):
            continue
        local_path = os.path.join(local_dir, fname)
        repo_path = f"clusters/{EMBEDDING_MODEL}/{corpus_type}/{fname}"

        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            commit_message=f"Upload cluster assignments for {corpus_type}/{fname} (by {user_info['name']})",
        )
        print(f"Uploaded {repo_path}")

### Summary: silhouette scores and cluster counts per category

One consolidated view for deciding, per category, whether k-means or HDBSCAN produced the more usable structure, and whether any category's clustering looks weak enough to warrant falling back to the cheap arxiv-label approach for that category specifically rather than forcing a clustering result that doesn't hold up.

In [ ]:
print(f"{'corpus':<12} {'category':<8} {'best_k':<7} {'kmeans_sil':<11} {'hdbscan_n':<10} {'hdbscan_sil':<12} {'hdbscan_noise'}")
print("-" * 80)

for (corpus_type, category), result in sorted(all_results.items()):
    best_k = result["best_k"]
    kmeans_sil = result["kmeans_sweep"][best_k]["silhouette"]

    hdbscan_n = "-"
    hdbscan_sil = "-"
    hdbscan_noise = "-"
    if result["hdbscan_result"] is not None:
        hr = result["hdbscan_result"]
        hdbscan_n = str(hr["n_clusters"])
        hdbscan_sil = f"{hr['silhouette']:.4f}" if hr["silhouette"] is not None else "n/a"
        hdbscan_noise = str(hr["n_noise"])

    print(f"{corpus_type:<12} {category:<8} {best_k:<7} {kmeans_sil:<11.4f} {hdbscan_n:<10} {hdbscan_sil:<12} {hdbscan_noise}")

print("\nNext step: read through the spot-check files under data/clusters/" + EMBEDDING_MODEL + "/spot_check/")
print("for each category, confirm the clusters look like coherent sub-themes before using them as W3 ground truth.")